# Preping Data for Analysis & Prediction

# A. ETL, Feature Engineering & EDA

## 1. Extracting data, normalizing column names & initial exploration

In [1]:
# Importing libraries/modules
import pandas as pd
import numpy as np

import getpass # Module - Enable password entry for access

from sqlalchemy import create_engine # makes a connection to sql server
from sqlalchemy.engine import URL # Converts into url text, prevents injection

In [2]:
# Extract from CSV
trasactions = pd.read_csv("dirty_financial_transactions.csv")

# Convert columns to lower case for normalization
trasactions.columns = trasactions.columns.str.lower()

#View final
trasactions.head()

,transaction_id,transaction_date,customer_id,product_name,quantity,price,payment_method,transaction_status
0,T0001,2024-08-02,C2205,Headphones,-5.0,$420.21,pay pal,NaN
1,T0002,2020-02-10,C3156,Coffee,469.0,-445.34202525395585,creditcard,Pending
2,T0003,2025-02-30,C2919,Tablet,-4.0,810.9930123946459,credit card,completed
3,T0004,2020-08-17,C3009,Tab,-7.0,868.6083413217348,PayPal,Pending
4,T0005,2025-02-30,C3488,Coffee Machine,-10.0,-763.1224490039416,PayPal,completed


In [3]:
print(trasactions.info()) # Checking null and data type
print()
print(trasactions.isna().sum()) # Checking NaN values

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 8 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   transaction_id      94982 non-null   object 
 1   transaction_date    95120 non-null   object 
 2   customer_id         95122 non-null   object 
 3   product_name        100000 non-null  object 
 4   quantity            94981 non-null   float64
 5   price               66503 non-null   object 
 6   payment_method      100000 non-null  object 
 7   transaction_status  83321 non-null   object 
dtypes: float64(1), object(7)
memory usage: 6.1+ MB
None

transaction_id         5018
transaction_date       4880
customer_id            4878
product_name              0
quantity               5019
price                 33497
payment_method            0
transaction_status    16679
dtype: int64


## 2. Cleaning transaction_id

In [4]:
# Comparing Index vs transaction_id values
tran_id = trasactions["transaction_id"].str.extract(r'(\d+)')[0].astype(float) # Extracting numberic value from id
filled_numeric = tran_id.copy() # Copy for safe computations

# Checking % of values where diff between current and previous value is 1
diff_relative = filled_numeric.diff() # Does current id - previous id for each id
total = len(diff_relative)-1 # index 0 diff will be NaN
valid = len(diff_relative[diff_relative == 1])
print("Percent of values where next values is (1 - current value) = ", round(valid*100/total),"%")

# Checking % of total values where index value and transaction value match
expected = filled_numeric.index + 1
correct = (expected == filled_numeric)
print("Percent of values where index and transaction id match = ", round(correct.sum()*100/len(correct)),"%")

# Checking % of non NaN values where index value and transaction value match
mask = filled_numeric.notna()
total = filled_numeric[mask].count()
print("Percent of non NaN values where index and transaction id match = ",round(correct.sum()*100/total), "%")

# Checking the 1% non NaN bad mismatches
filled_numeric[mask][expected[mask] != filled_numeric[mask]][400:500] # Random mismatch, not systemic mismatch

# Finding all non NaN mismatches
mismatches = filled_numeric[mask][filled_numeric[mask] != expected[mask]]
print("Total Non-NaN mismatches:", mismatches.count())
print()

# Converting non NaN mismatches to NaN
filled_numeric[(mask) & (filled_numeric[mask] != expected[mask])] = np.nan
print(filled_numeric.info())
print()

# Filling all NaN with actual values
cool = pd.Series(expected)
filled_numeric = filled_numeric.fillna(cool)

# Changing numbers to transactions
trasactions["transaction_idNew"] = ("T" + filled_numeric.astype(int).astype(str).str.zfill(4))

# Checking if it works
trasactions[trasactions.index + 1 != trasactions["transaction_idNew"].str.extract(r'(\d+)')[0].astype(int)].count()
len(trasactions[trasactions.index + 1 != trasactions["transaction_idNew"].str.extract(r'(\d+)')[0].astype(int)])

Percent of values where next values is (1 - current value) =  88 %
Percent of values where index and transaction id match =  94 %
Percent of non NaN values where index and transaction id match =  99 %
Total Non-NaN mismatches: 945

<class 'pandas.core.series.Series'>
RangeIndex: 100000 entries, 0 to 99999
Series name: 0
Non-Null Count  Dtype  
--------------  -----  
94037 non-null  float64
dtypes: float64(1)
memory usage: 781.4 KB
None



0

## 3. Cleaning transaction_dates

### 3.1 EDA on transaction_dates

In [5]:
# Converting object to datetime and checking NaT
trasactions["date_parsed"] = pd.to_datetime(trasactions['transaction_date'], format= "%Y-%m-%d", errors='coerce')
print("Error Dates:", trasactions["date_parsed"].isna().sum()) 
print()

# Checking Invalid dates
error_dates = trasactions[trasactions["date_parsed"].isna()]["transaction_date"]
print(error_dates.value_counts(dropna = False))
print()

# Comparing with valid dates
print(trasactions["date_parsed"].dt.year.unique())

Error Dates: 68261

transaction_date
2023-13-01    31834
2025-02-30    31547
NaN            4880
Name: count, dtype: int64

[2024. 2020.   nan 2021. 2023. 2022. 2025.]


### 3.2 Transforming Non-NaN transaction_dates to 'YYYY-MM-DD' format

In [6]:
# Creating function for replacing invalid dates with valid
def dates_conversion(dates):
    try:
        change = pd.to_datetime(dates, format= "%Y-%m-%d")
        return change
    except:
        parts = dates.split('-')
        if int(parts[1]) > 12:
            return pd.to_datetime(f'{parts[0]}-{parts[2]}-{parts[1]}')
        elif int(parts[1]) == 2 and int(parts[2]) > 28:
            return pd.to_datetime(f'{parts[0]}-{parts[1]}-28')
        else:
            return pd.NaT

# Replacing invalid dates with valid
trasactions["clean_dates"] = trasactions["transaction_date"].apply(dates_conversion)

# 1st check
trasactions.isna().sum()

transaction_id         5018
transaction_date       4880
customer_id            4878
product_name              0
quantity               5019
price                 33497
payment_method            0
transaction_status    16679
transaction_idNew         0
date_parsed           68261
clean_dates            4880
dtype: int64

### 3.3 Transforming NaN dates to most common date

In [7]:
# Checking trends of dates
print(trasactions["clean_dates"].value_counts())
print()

# Replacing NaT dates with Mode
trasactions["clean_dates"] = trasactions["clean_dates"].fillna(trasactions["clean_dates"].mode()[0])

# Dropping prased dates column
trasactions = trasactions.drop(columns= "date_parsed")

# Final check
print(trasactions.isna().sum())

clean_dates
2023-01-13    31849
2025-02-28    31547
2023-10-16       33
2020-05-18       30
2022-07-13       30
              ...  
2021-05-17        7
2023-07-17        7
2022-06-12        7
2020-10-13        6
2022-04-30        6
Name: count, Length: 1860, dtype: int64

transaction_id         5018
transaction_date       4880
customer_id            4878
product_name              0
quantity               5019
price                 33497
payment_method            0
transaction_status    16679
transaction_idNew         0
clean_dates               0
dtype: int64


### 3.4 Adding a weekday feature

In [ ]:
# Putting a weekday feature
trasactions["weekday"] = trasactions['clean_dates'].dt.weekday

## 4. Cleaning product_name

In [9]:
# Finding unique values to check for discrepancies
print(sorted(trasactions["product_name"].unique())) # Finding Unique values
print()

# Cleaning keystroke errors
conditions = [
    trasactions["product_name"].isin(['C', 'Co', 'Cof', 'Coff', 'Coffe', 'Coffee', 'Coffee ', 'Coffee M', 'Coffee Ma', 'Coffee Mac', 
                                      'Coffee Mach', 'Coffee Machi', 'Coffee Machin', 'Coffee Machine']),
    trasactions["product_name"].isin(['H', 'He', 'Hea', 'Head', 'Headp', 'Headph', 'Headpho', 'Headphon', 'Headphone', 'Headphones']),
    trasactions["product_name"].isin(['L', 'La', 'Lap', 'Lapt', 'Lapto', 'Laptop']),
    trasactions["product_name"].isin(['S', 'Sm', 'Sma', 'Smar', 'Smart', 'Smartp', 'Smartph', 'Smartpho', 'Smartphon', 'Smartphone']),
    trasactions["product_name"].isin(['T', 'Ta', 'Tab', 'Tabl', 'Table', 'Tablet'])
]

choices = ['Coffee Machine', 'Headphones', 'Laptop', 'Smartphone', 'Tablet']
trasactions["product_name_cleaned"] = np.select(conditions, choices, default="Others")

trasactions["product_name_cleaned"].value_counts() # Final Check

['C', 'Co', 'Cof', 'Coff', 'Coffe', 'Coffee', 'Coffee ', 'Coffee M', 'Coffee Ma', 'Coffee Mac', 'Coffee Mach', 'Coffee Machi', 'Coffee Machin', 'Coffee Machine', 'H', 'He', 'Hea', 'Head', 'Headp', 'Headph', 'Headpho', 'Headphon', 'Headphone', 'Headphones', 'L', 'La', 'Lap', 'Lapt', 'Lapto', 'Laptop', 'S', 'Sm', 'Sma', 'Smar', 'Smart', 'Smartp', 'Smartph', 'Smartpho', 'Smartphon', 'Smartphone', 'T', 'Ta', 'Tab', 'Tabl', 'Table', 'Tablet']



product_name_cleaned
Tablet            20151
Smartphone        20067
Laptop            19980
Headphones        19934
Coffee Machine    19868
Name: count, dtype: int64

## 5. Cleaning quantity & price

### 5.1 EDA and transforming negative values

In [10]:
# Converting price from object to float
trasactions['price'] = trasactions["price"].str.replace("$", "").str.strip().astype(float).round(2)
print(trasactions['price'].dtype) # checking data type

# Checking combo of negative values across quantity and price
print(trasactions[trasactions["quantity"] < 0]["transaction_status"].value_counts(dropna = False))
print()
print(trasactions[trasactions["price"] < 0]["transaction_status"].value_counts(dropna = False))
print()
print(trasactions[(trasactions["quantity"] < 0) | (trasactions["price"] < 0)]["transaction_status"].value_counts(dropna = False))

# Conclusion - Since consistent accross all 3 transaction states, consider signage issue
# Converting all to positive
trasactions["quantity"] = trasactions["quantity"].abs()
trasactions["price"] = trasactions["price"].abs()

float64
transaction_status
Completed    5302
Pending      5288
complete     5283
completed    5278
Failed       5255
NaN          5213
Name: count, dtype: int64

transaction_status
complete     5572
Pending      5564
NaN          5544
completed    5540
Completed    5531
Failed       5444
Name: count, dtype: int64

transaction_status
Pending      9128
completed    9093
complete     9091
NaN          9070
Completed    9049
Failed       9007
Name: count, dtype: int64


### 5.2 Finding & removing outliners, and substituting them with median within each product category

In [11]:
# Function to find outliners for each product
def nul_outliners(group):
    Q1 = group.quantile(0.25)
    Q3 = group.quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5*IQR
    upper = Q3 + 1.5*IQR

    return group.where((group > lower) & (group < upper))

# Converting Outliners to NaN and substituing NaN with median of each product
for col in ["quantity", "price"]:
    trasactions[f'{col}_new'] = trasactions.groupby("product_name")[col].transform(nul_outliners)

    trasactions[f'{col}_new'] = trasactions[f'{col}_new'].fillna(trasactions.groupby('product_name')[f'{col}_new'].transform('median'))

# Converting Quantity to integer
trasactions.quantity_new = trasactions.quantity_new.astype(int)

# Final Check
print(trasactions.dtypes)
print()
print(trasactions.isna().sum())

transaction_id                  object
transaction_date                object
customer_id                     object
product_name                    object
quantity                       float64
price                          float64
payment_method                  object
transaction_status              object
transaction_idNew               object
clean_dates             datetime64[ns]
weekday                          int32
product_name_cleaned            object
quantity_new                     int64
price_new                      float64
dtype: object

transaction_id           5018
transaction_date         4880
customer_id              4878
product_name                0
quantity                 5019
price                   33497
payment_method              0
transaction_status      16679
transaction_idNew           0
clean_dates                 0
weekday                     0
product_name_cleaned        0
quantity_new                0
price_new                   0
dtype: int64


## 6. Cleaning payment_method & transaction_status

### 6.1 Cleaning payment_method

In [12]:
# Finding unique values
print(trasactions["payment_method"].unique())
print()

#Trimming and making it proper words
trasactions["payment_method"] = trasactions["payment_method"].str.strip().str.title()

print(trasactions["payment_method"].unique()) # Checking again
print()

# Fixing PayPal
trasactions.loc[trasactions["payment_method"].isin(['Pay Pal', 'Paypal']), "payment_method"] = 'PayPal'

print(trasactions["payment_method"].unique()) # Checking again
print()

# Fixing Credit card
trasactions.loc[trasactions["payment_method"] == "Creditcard", 'payment_method'] = 'Credit Card'

print(trasactions["payment_method"].unique()) # Final Check

['pay pal' 'creditcard' 'credit card' 'PayPal' 'Cash' 'PayPal '
 'Credit Card']

['Pay Pal' 'Creditcard' 'Credit Card' 'Paypal' 'Cash']

['PayPal' 'Creditcard' 'Credit Card' 'Cash']

['PayPal' 'Credit Card' 'Cash']


### 6.2 Cleaning transaction_status

In [13]:
# Finding Unique values
print(trasactions["transaction_status"].unique())
print()

#Trimming and making it proper words
trasactions["transaction_status"] = trasactions["transaction_status"].str.strip().str.title()

print(trasactions["transaction_status"].unique()) # Finding Unique values
print()

# Fixing complete
trasactions.loc[trasactions["transaction_status"] == 'Complete', "transaction_status"] = 'Completed'

print(trasactions["transaction_status"].unique()) # Finding Unique values
print()

# Checking NaN
print(trasactions.transaction_status.value_counts(dropna=False))

# Final Check
trasactions.isna().sum()

[nan 'Pending' 'completed' 'Completed' 'complete' 'Failed']

[nan 'Pending' 'Completed' 'Complete' 'Failed']

[nan 'Pending' 'Completed' 'Failed']

transaction_status
Completed    49931
Failed       16795
NaN          16679
Pending      16595
Name: count, dtype: int64


transaction_id           5018
transaction_date         4880
customer_id              4878
product_name                0
quantity                 5019
price                   33497
payment_method              0
transaction_status      16679
transaction_idNew           0
clean_dates                 0
weekday                     0
product_name_cleaned        0
quantity_new                0
price_new                   0
dtype: int64

## 7. Finding and removing duplicate transactions

In [14]:
# Defining the columns that define a duplicate
dupcond = [
    'customer_id', 'product_name_cleaned', 'quantity_new', 'price_new', 
    'clean_dates', 'payment_method', 'transaction_status'
]

# Counting duplicates
duplicated_count = trasactions.duplicated(subset=dupcond).sum()
print('No of duplicates =', duplicated_count)
trasactions[trasactions.duplicated(subset=dupcond) == True].sort_values(by = ['quantity_new', 'transaction_date']) # Visualizing duplicates

# Removing duplicates
trasactions = trasactions.drop_duplicates(subset = dupcond, keep='first')

# Checking duplicates again
trasactions.duplicated(subset=dupcond).sum()

No of duplicates = 1278


np.int64(0)

In [15]:
# Final Check
trasactions.info()

<class 'pandas.core.frame.DataFrame'>
Index: 98722 entries, 0 to 99998
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   transaction_id        93770 non-null  object        
 1   transaction_date      93921 non-null  object        
 2   customer_id           94141 non-null  object        
 3   product_name          98722 non-null  object        
 4   quantity              93792 non-null  float64       
 5   price                 65875 non-null  float64       
 6   payment_method        98722 non-null  object        
 7   transaction_status    82228 non-null  object        
 8   transaction_idNew     98722 non-null  object        
 9   clean_dates           98722 non-null  datetime64[ns]
 10  weekday               98722 non-null  int32         
 11  product_name_cleaned  98722 non-null  object        
 12  quantity_new          98722 non-null  int64         
 13  price_new            

# B. Preping final data and pushing them into MySQL

## 1. Extracting cleaned columns into a final dataset

In [16]:
# Extracting clean columns
transactions_cleaned = trasactions.iloc[:, [8, 9, 10, 2, 11, 12, 13, 6, 7]].copy()
print(transactions_cleaned.info())
print()

# Renaming columns
transactions_cleaned = transactions_cleaned.rename(columns={
    "transaction_idNew" : "transaction_id",
    "clean_dates" : "transaction_date",
    "product_name_cleaned" : "product_name",
    "quantity_new" : "quantity",
    "price_new" : "price"
})
print(transactions_cleaned.columns)

# Checking NaN values
transactions_cleaned.isna().sum()

<class 'pandas.core.frame.DataFrame'>
Index: 98722 entries, 0 to 99998
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   transaction_idNew     98722 non-null  object        
 1   clean_dates           98722 non-null  datetime64[ns]
 2   weekday               98722 non-null  int32         
 3   customer_id           94141 non-null  object        
 4   product_name_cleaned  98722 non-null  object        
 5   quantity_new          98722 non-null  int64         
 6   price_new             98722 non-null  float64       
 7   payment_method        98722 non-null  object        
 8   transaction_status    82228 non-null  object        
dtypes: datetime64[ns](1), float64(1), int32(1), int64(1), object(5)
memory usage: 7.2+ MB
None

Index(['transaction_id', 'transaction_date', 'weekday', 'customer_id',
       'product_name', 'quantity', 'price', 'payment_method',
       'transaction_status'],

transaction_id            0
transaction_date          0
weekday                   0
customer_id            4581
product_name              0
quantity                  0
price                     0
payment_method            0
transaction_status    16494
dtype: int64

## 2. Splitting the data into Audit, Historical and Pending datasets

In [21]:
# Splitting cleaned data

customer_nan = transactions_cleaned["customer_id"].isna()
transactions_nan = transactions_cleaned["transaction_status"].isna()

# Creating audit data for operational team
df_audit = transactions_cleaned[customer_nan | transactions_nan].copy()
df_audit.isna().sum()

# Creating pending data, used for prediction
df_pending = transactions_cleaned[(transactions_cleaned["transaction_status"] == "Pending") & (~customer_nan)].copy()
print(df_pending.isna().sum())
print()

# Creating historical data for creating the regression model
df_model = transactions_cleaned[(transactions_cleaned["transaction_status"].isin(["Completed", "Failed"])) & (~customer_nan)].copy()
print(df_model.info())

transaction_id        0
transaction_date      0
weekday               0
customer_id           0
product_name          0
quantity              0
price                 0
payment_method        0
transaction_status    0
dtype: int64

<class 'pandas.core.frame.DataFrame'>
Index: 62766 entries, 2 to 99998
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   transaction_id      62766 non-null  object        
 1   transaction_date    62766 non-null  datetime64[ns]
 2   weekday             62766 non-null  int32         
 3   customer_id         62766 non-null  object        
 4   product_name        62766 non-null  object        
 5   quantity            62766 non-null  int64         
 6   price               62766 non-null  float64       
 7   payment_method      62766 non-null  object        
 8   transaction_status  62766 non-null  object        
dtypes: datetime64[ns](1), float64(1), int32(1), in

## 3. Pushing data to a MySQL database

In [22]:
# Creating url link and connecting to engine
db_url = URL.create(
    drivername="mysql+mysqlconnector",
    username="root",
    password=getpass.getpass("Enter database password:"), # Secure way to collect password
    host="localhost",
    port=3306,
    database='transactions'
)
engine = create_engine(db_url)

Enter database password: ········


In [23]:
# Pushing data to SQL database
df_audit.to_sql("data_quality_audit", engine, if_exists='replace', index=False)
df_pending.to_sql("pending", engine, if_exists='replace', index=False)
df_model.to_sql("historical", engine, if_exists='replace', index=False)

62766

# C. Assumptions
1. A transaction is considered duplicate if the same customer brought the same quantity of same items on the same day using same payment method
2. For price/quantity mismatch, substitute with median for correct representation of data and to neither undervalue or overvalue the transaction